In [0]:
%pip install yfinance

In [0]:
# ============================================================
# YFINANCE INGESTION PIPELINE
#  Raw Data Collector
# Stores data in Unity Catalog Volume
# ============================================================

import yfinance as yf
import json
import time
import random
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed


# ------------------------------------------------------------
# BASE PATH (Unity Catalog Volume)
# ------------------------------------------------------------
BASE_PATH = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/"

FOLDERS = [
    "stock",
    "news",
    "income_statement",
    "balance_sheet",
    "cashflow",
    "info"
]

# create folders if not exist
for f in FOLDERS:
    os.makedirs(BASE_PATH + f, exist_ok=True)


# ------------------------------------------------------------
# COMPANY LIST (single source of truth)
# ------------------------------------------------------------
companies = [
    {"name": "J SAINSBURY PLC", "ticker": "SBRY.L"},
    {"name": "JD SPORTS FASHION PLC", "ticker": "JD.L"},
    {"name": "OCADO GROUP PLC", "ticker": "OCDO.L"},
    {"name": "LLOYDS BANKING GROUP PLC", "ticker": "LLOY.L"},
    {"name": "NATWEST GROUP PLC", "ticker": "NWG.L"},
    {"name": "NATIONAL GRID PLC", "ticker": "NG.L"},
    {"name": "SSE PLC", "ticker": "SSE.L"},
    {"name": "DRAX GROUP PLC", "ticker": "DRX.L"},
    {"name": "BALFOUR BEATTY PLC", "ticker": "BBY.L"},
    {"name": "PERSIMMON PLC", "ticker": "PSN.L"},
    {"name": "EASYJET PLC", "ticker": "EZJ.L"},
    {"name": "INTERCONTINENTAL HOTELS GROUP PLC", "ticker": "IHG.L"},
    {"name": "HIKMA PHARMACEUTICALS PLC", "ticker": "HIK.L"},
    {"name": "PEARSON PLC", "ticker": "PSON.L"}, 
    {"name": "BT GROUP PLC", "ticker": "BT-A.L"},
    {"name": "ITV PLC", "ticker": "ITV.L"},
    {"name": "JOHNSON MATTHEY PLC", "ticker": "JMAT.L"},
    {"name": "MELROSE INDUSTRIES PLC", "ticker": "MRO.L"},
    {"name": "AVIVA PLC", "ticker": "AV.L"},
    {"name": "ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC", "ticker": "AML.L"},
]


# ------------------------------------------------------------
# SAFE DATAFRAME CONVERTER
# fixes Timestamp + NaN issues
# ------------------------------------------------------------
def safe_df_to_records(df):
    if df is None or df.empty:
        return []

    df = df.copy()
    df.columns = [str(c) for c in df.columns]
    df = df.reset_index()
    df = df.where(pd.notnull(df), None)

    return df.to_dict(orient="records")


# ------------------------------------------------------------
# SAFE JSON WRITER
# always writes file (no missing files anymore)
# ------------------------------------------------------------
def save_json(folder, filename, data):
    path = f"{BASE_PATH}{folder}/{filename}"

    if data is None:
        data = []

    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)


# ------------------------------------------------------------
# INGEST SINGLE COMPANY
# ------------------------------------------------------------
def ingest_company(company):

    ticker = company["ticker"]
    name = company["name"]

    # normalize ticker for filename safety
    safe_ticker = ticker.replace(".", "_").replace("-", "_")

    print(f"Processing {name} ({ticker})")

    try:
        stock = yf.Ticker(ticker)

        # ---------------- STOCK ----------------
        try:
            hist = stock.history(period="1y")
            save_json("stock", f"{safe_ticker}.json", safe_df_to_records(hist))
        except Exception:
            pass

        # ---------------- NEWS ----------------
        try:
            save_json("news", f"{safe_ticker}.json", stock.news)
        except Exception:
            pass

        # ---------------- INCOME STATEMENT ----------------
        try:
            income = stock.financials
            save_json("income_statement", f"{safe_ticker}.json", safe_df_to_records(income))
        except Exception:
            pass

        # ---------------- BALANCE SHEET ----------------
        try:
            bs = stock.balance_sheet
            save_json("balance_sheet", f"{safe_ticker}.json", safe_df_to_records(bs))
        except Exception:
            pass

        # ---------------- CASHFLOW ----------------
        try:
            cf = stock.cashflow
            save_json("cashflow", f"{safe_ticker}.json", safe_df_to_records(cf))
        except Exception:
            pass

        # ---------------- INFO ----------------
        try:
            save_json("info", f"{safe_ticker}.json", stock.info)
        except Exception:
            pass

    except Exception as e:
        print("FAILED:", ticker, e)

    # avoid rate limits
    time.sleep(1.2 + random.random())


# ------------------------------------------------------------
# PARALLEL EXECUTION
# ------------------------------------------------------------
def run_pipeline():
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(ingest_company, c) for c in companies]

        for f in as_completed(futures):
            try:
                f.result()
            except Exception as e:
                print("Worker error:", e)


# ------------------------------------------------------------
# RUN INGESTION
# ------------------------------------------------------------
run_pipeline()
print("Ingestion Cpmpleted")

In [0]:
# ------------------------------------------------------------
# SIMPLE HEALTH CHECK
# ------------------------------------------------------------

expected = set(
    [c["ticker"].replace(".", "_").replace("-", "_") for c in companies]
)

print("\nINGESTION CHECK\n")

for folder in FOLDERS:
    path = BASE_PATH + folder + "/"
    files = os.listdir(path)

    existing = set([f.replace(".json", "") for f in files])

    missing = expected - existing

    print(folder)
    print("files:", len(files))
    print("missing:", len(missing))

In [0]:
from collections import defaultdict

BASE_PATH = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/"

companies = [
    "SBRY_L", "JD_L", "OCDO_L", "LLOY_L", "NWG_L",
    "NG_L", "SSE_L", "DRX_L", "BBY_L", "PSN_L",
    "EZJ_L", "IHG_L", "HIK_L", "PSON_L",
    "BT_A_L", "ITV_L", "JMAT_L", "MRO_L",
    "AV_L", "AML_L"
]

datasets = [
    "stock",
    "news",
    "income_statement",
    "balance_sheet",
    "cashflow",
    "info"
]

report = defaultdict(dict)

# ------------------------------------------------------------
# CHECK FILES
# ------------------------------------------------------------
for dataset in datasets:

    path = f"{BASE_PATH}{dataset}/"

    try:
        files = dbutils.fs.ls(path)
        existing_files = [f.name.replace(".json", "") for f in files]

        missing = [c for c in companies if c not in existing_files]

        report[dataset]["total_files"] = len(existing_files)
        report[dataset]["missing_files"] = missing
        report[dataset]["status"] = "PASS" if len(missing) == 0 else "FAIL"

    except Exception as e:
        report[dataset]["status"] = "PATH_NOT_FOUND"
        report[dataset]["error"] = str(e)

# ------------------------------------------------------------
# PRINT REPORT
# ------------------------------------------------------------
for dataset, info in report.items():

    print("\n" + "="*50)
    print("DATASET:", dataset)
    print("STATUS:", info.get("status"))

    if "total_files" in info:
        print("Total files:", info["total_files"])

    if "missing_files" in info:
        print("Missing files:", info["missing_files"])